# M25 E2E — Тест цепочки image → image → video

**Порядок:** Ячейки запускать строго сверху вниз.

**Если GPU отключился:** Перезапусти рантайм, запусти ячейки 3→4 заново.

In [ ]:
#@title 1. Клонировать агента из GitHub
import os
import sys

AGENT_DIR = '/content/agent'
REPO_URL = 'https://github.com/Ovladimirovich/comfyui-agent.git'

# Удаляем предыдущий клон если есть
if os.path.exists(AGENT_DIR):
    !rm -rf {AGENT_DIR}
    print('Удалён предыдущий клон')

# Клонируем
!git clone {REPO_URL} {AGENT_DIR}

# Добавляем в путь
sys.path.insert(0, AGENT_DIR)

# Проверяем структуру
dirs = ['app', 'tests', 'workflows']
ok = all(os.path.isdir(os.path.join(AGENT_DIR, d)) for d in dirs)
if ok:
    print('Агент готов:', AGENT_DIR)
else:
    print('ОШИБКА: не найдены нужные папки')

In [ ]:
#@title 2. Установить зависимости + запустить ComfyUI
import os
import subprocess
import time
import json
import urllib.request

COMFY_DIR = '/content/ComfyUI'
AGENT_DIR = '/content/agent'

# Клонируем ComfyUI если нужно
if not os.path.exists(COMFY_DIR):
    print('Клонирую ComfyUI...')
    !git clone https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}
    print('Устанавливаю зависимости ComfyUI...')
    !pip install -r {COMFY_DIR}/requirements.txt -q
else:
    print('ComfyUI уже есть')

# Устанавливаем зависимости агента
req = os.path.join(AGENT_DIR, 'requirements.txt')
if os.path.exists(req):
    print('Устанавливаю зависимости агента...')
    !pip install -r {AGENT_DIR}/requirements.txt -q
else:
    print('requirements.txt нет, пропускаю')

# Функция проверки
def comfy_ready():
    try:
        urllib.request.urlopen('http://localhost:8188/system_stats', timeout=5)
        return True
    except:
        return False

# Запускаем ComfyUI если не запущен
if comfy_ready():
    print('ComfyUI уже запущен на :8188')
else:
    print('Запускаю ComfyUI...')
    proc = subprocess.Popen(
        ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
        cwd=COMFY_DIR,
        stdout=open('/tmp/comfyui.log', 'w'),
        stderr=subprocess.STDOUT
    )
    for i in range(60):
        time.sleep(2)
        if comfy_ready():
            print(f'ComfyUI готов за {i*2}с')
            break
    else:
        print('ComfyUI не запустился за 120с')
        print(open('/tmp/comfyui.log').read()[-500:])

# Показываем статистику
if comfy_ready():
    stats = json.loads(urllib.request.urlopen('http://localhost:8188/system_stats').read())
    device = stats['devices'][0]
    print(f'GPU: {device["name"]}')
    print(f'VRAM: {device["vram_total"]//1024**3}GB / {device["vram_free"]//1024**3}GB свободно')
    print(f'ComfyUI: {stats["system"]["comfyui_version"]}')

In [ ]:
#@title 3. Запустить туннель + найти URL
import os
import re
import subprocess
import time
import json
import urllib.request

LOG = '/tmp/cloudflared.log'

# Убиваем старый туннель
!pkill cloudflared 2>/dev/null || true
time.sleep(2)

# Запускаем новый
print('Запускаю туннель...')
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=open(LOG, 'w'),
    stderr=subprocess.STDOUT
)

# Ждём URL
url = None
for i in range(30):
    time.sleep(2)
    try:
        text = open(LOG).read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
        if match:
            url = match.group(0)
            break
    except:
        pass

if url:
    os.environ['COMFY_REMOTE_URL'] = url
    # Проверяем что туннель работает
    time.sleep(3)
    try:
        r = urllib.request.urlopen(f'{url}/system_stats', timeout=15)
        stats = json.loads(r.read())
        device = stats['devices'][0]
        print(f'Туннель готов!')
        print(f'URL: {url}')
        print(f'GPU: {device["name"]}')
    except Exception as e:
        print(f'URL найден, но пока не доступен: {e}')
        print('Подожди 10с и запусти ячейку заново')
else:
    print('URL не найден. Логи:')
    print(open(LOG).read()[-500:])

In [ ]:
#@title 4. Запустить M25 E2E тест
import os

url = os.environ.get('COMFY_REMOTE_URL')

if not url:
    print('URL не задан. Запусти ячейку 3 заново.')
else:
    print(f'Цель: {url}')
    print('Запускаю M25 E2E...')
    print('=' * 60)
    !cd /content/agent && python tests/_m25_e2e_runner.py --url {url}